# SynBioCrow 2.4.22 — Pairwise Ranking Suppression Audit
Upload only the 2.4.22 SOURCE ZIP. Returns to frozen 2.4.16 ranking and diagnoses why the best literature-near candidates are outranked. 2.4.21 is retained as a negative experiment.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
drive.mount("/content/drive",force_remount=False);T0=time.time()
BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4");REPO=Path("/content/SynBioCrow_2_4_22");SRC=Path("/content/src2422");RUN=Path("/content/run2422");OUT=BASE/"pairwise_suppression_2_4_22";OUT.mkdir(parents=True,exist_ok=True)
def st(p,m):print(f"[2.4.22] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)
def run(cmd,cwd=None,timeout=900):
 e=os.environ.copy()
 if cwd:e["PYTHONPATH"]=str(cwd)+os.pathsep+e.get("PYTHONPATH","")
 print("$"," ".join(map(str,cmd)),flush=True);q=subprocess.run(list(map(str,cmd)),cwd=cwd,env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout);print(q.stdout,flush=True)
 if q.returncode:raise subprocess.CalledProcessError(q.returncode,cmd,output=q.stdout)
st(3,"upload SOURCE ZIP");u=files.upload()
if len(u)!=1:raise RuntimeError("upload exactly one SOURCE ZIP")
name=next(iter(u));z=Path("/content")/name;z.write_bytes(u[name])
st(10,"clone + overlay");shutil.rmtree(REPO,ignore_errors=True);run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
shutil.rmtree(SRC,ignore_errors=True);SRC.mkdir(parents=True)
with zipfile.ZipFile(z) as q:q.extractall(SRC)
for p in SRC.rglob("*"):
 if p.is_file() and p.name not in {"README_SOURCE.txt","SOURCE_MANIFEST.json"}:
  d=REPO/p.relative_to(SRC);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
st(25,"install core");run([sys.executable,"-m","pip","install","-q","-e",str(REPO)])
ranked=BASE/"native_discrimination_2_4_16"/"native_enriched_ranked_routes.json";audit=BASE/"internal_anchor_audit_2_4_18_1"/"corrected_internal_anchor_audit.json";dec=BASE/"failure_decomposition_2_4_19"/"failure_decomposition_summary.json";failed=BASE/"consensus_rerank_2_4_21"/"consensus_rerank_summary.json"
for p in (ranked,audit,dec,failed):
 if not p.is_file():raise FileNotFoundError(p)
st(45,"audit pairwise suppression under frozen 2.4.16 ranking");shutil.rmtree(RUN,ignore_errors=True);RUN.mkdir(parents=True)
run([sys.executable,str(REPO/"scripts/v24_pairwise_suppression_2_4_22.py"),"--ranked",str(ranked),"--audit",str(audit),"--decomposition",str(dec),"--failed-policy-summary",str(failed),"--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),"--out-dir",str(RUN)],cwd=REPO)
st(75,"verify negative experiment + firewall");s=json.loads((RUN/"pairwise_suppression_audit.json").read_text());assert s["failed_2_4_21_policy"]["all_worsened"] and not s["ranking_changed"] and not s["generation_invoked"] and not s["validation_truth_accessed"] and not s["evaluation_truth_accessed"];print(json.dumps(s,indent=2))
st(88,"persist + bundle")
for p in RUN.iterdir():
 if p.is_file():shutil.copy2(p,OUT/p.name)
bundle=OUT/"SynBioCrow_2_4_22_PAIRWISE_SUPPRESSION_AUDIT_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as q:
 for p in sorted(OUT.iterdir()):
  if p.is_file() and p!=bundle:q.write(p,p.name)
st(100,"PASS");print("BUNDLE",bundle);print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest());files.download(str(bundle))


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/pairwise_suppression_2_4_22/SynBioCrow_2_4_22_PAIRWISE_SUPPRESSION_AUDIT_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
